# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Duchalsoham12/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [85]:
import pandas as pd
import numpy as np
import os

file_path = "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Target column:", "is_declining_label")
print("Classes:")
print(df["is_declining_label"].value_counts())

Target column: is_declining_label
Classes:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [86]:
 from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]) & set(test["client_id"])
print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


In [87]:
excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

feature_cols = [
    c for c in df.columns
    if c not in excluded
]

X_train = train[feature_cols]
y_train = train["is_declining_label"]

X_test = test[feature_cols]
y_test = test["is_declining_label"]

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Total features:", len(feature_cols))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Total features: 40
Numeric features: 29
Categorical features: 11


In [88]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [89]:
pred_prob = model.predict_proba(X_test)[:, 1]

def precision_at_50(y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_n = min(50, len(y_true))
    top_indices = np.argsort(scores)[::-1][:top_n]

    return y_true[top_indices].mean()

model_p50 = precision_at_50(y_test, pred_prob)

print(
    "Random Forest Precision@50:",
    round(model_p50, 3)
)

Random Forest Precision@50: 1.0


In [90]:
# Week-4 baseline evaluated on the SAME held-out test set

baseline_scores = (
    test["ctr"].fillna(0)
    + test["engagement_rate"].fillna(0)
)

baseline_p50_same_split = precision_at_50(
    y_test,
    baseline_scores
)

print(
    "Week-4 baseline Precision@50 on test split:",
    round(baseline_p50_same_split, 3)
)

Week-4 baseline Precision@50 on test split: 0.7


In [91]:
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "precision_at_50": [
        baseline_p50_same_split,
        model_p50
    ]
})

display(comparison)

,method,precision_at_50
0,Week-4 baseline,0.7
1,Random Forest,1.0


In [92]:
print("Evaluation rows:", len(test))
print("Same test set used for both methods:", True)
print("Random Forest Precision@50:", round(model_p50, 3))
print(
    "Baseline Precision@50:",
    round(baseline_p50_same_split, 3)
)

Evaluation rows: 6163
Same test set used for both methods: True
Random Forest Precision@50: 1.0
Baseline Precision@50: 0.7


In [93]:
error_analysis = test[
    [
        "content_id",
        "trend_direction",
        "trend_pct",
        "content_age_days",
        "days_since_last_update",
        "engagement_rate",
        "ai_traffic_pct"
    ]
].copy()

error_analysis["predicted_probability"] = pred_prob

error_analysis["predicted_class"] = (
    error_analysis["predicted_probability"] >= 0.5
).astype(int)

error_analysis["actual_class"] = test["is_declining_label"].values

error_analysis["correct"] = (
    error_analysis["predicted_class"]
    == error_analysis["actual_class"]
)

print("Test rows:", len(error_analysis))
print(
    "Incorrect predictions:",
    int((~error_analysis["correct"]).sum())
)

print("\nHighest-scored incorrect predictions:")

display(
    error_analysis[
        ~error_analysis["correct"]
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
    .head(10)
)

Test rows: 6163
Incorrect predictions: 1228

Highest-scored incorrect predictions:


,content_id,trend_direction,trend_pct,content_age_days,days_since_last_update,engagement_rate,ai_traffic_pct,predicted_probability,predicted_class,actual_class,correct
12332,content_4d9f36001f06,stable,-17.0,275,104,33.33,0.00,0.896667,1,0,False
2357,content_8f1409b2674e,stable,-17.9,271,104,0.00,0.00,0.890000,1,0,False
20736,content_41baf0722ad9,stable,-14.3,275,104,0.00,0.00,0.883333,1,0,False
10080,content_35d63627bf3e,stable,8.7,238,103,0.00,0.00,0.846667,1,0,False
5399,content_6677fd6c4ea5,stable,-13.7,280,104,0.00,0.00,0.843333,1,0,False
11061,content_0b47dae0c7f9,stable,-13.3,238,103,0.00,0.00,0.826667,1,0,False
1517,content_816d77e36e14,stable,-19.1,275,104,0.00,0.00,0.813333,1,0,False
1439,content_5585a0e7089c,stable,-8.7,223,102,0.00,0.00,0.810000,1,0,False
28582,content_f49660e074e9,stable,3.1,223,102,6.25,6.25,0.806667,1,0,False
22526,content_1d0963b56227,up,43.6,280,104,20.00,0.00,0.806667,1,0,False


In [94]:
print("ML-08 completed successfully.")
print("Train rows:", len(train))
print("Test rows:", len(test))
print(
    "Client overlap:",
    len(set(train["client_id"]) & set(test["client_id"]))
)
print(
    "Random Forest Precision@50:",
    round(model_p50, 3)
)

ML-08 completed successfully.
Train rows: 23837
Test rows: 6163
Client overlap: 0
Random Forest Precision@50: 1.0


I chose Random Forest because the target is binary: declining or not declining. It can handle different types of features and non-linear relationships, making it suitable for this dataset. I will compare its measured performance with the Week-4 baseline using the same evaluation approach

In [95]:
import os

for root, dirs, files in os.walk("flyrank-ml-internship"):
    for file in files:
        print(os.path.join(root, file))

flyrank-ml-internship/CLAUDE.md
flyrank-ml-internship/LICENSE
flyrank-ml-internship/README.md
flyrank-ml-internship/index.md
flyrank-ml-internship/SETUP.md
flyrank-ml-internship/.gitignore
flyrank-ml-internship/DATA_USE.md
flyrank-ml-internship/requirements.txt
flyrank-ml-internship/GUIDE.md
flyrank-ml-internship/AGENTS.md
flyrank-ml-internship/.git/index
flyrank-ml-internship/.git/config
flyrank-ml-internship/.git/HEAD
flyrank-ml-internship/.git/packed-refs
flyrank-ml-internship/.git/description
flyrank-ml-internship/.git/hooks/pre-commit.sample
flyrank-ml-internship/.git/hooks/push-to-checkout.sample
flyrank-ml-internship/.git/hooks/commit-msg.sample
flyrank-ml-internship/.git/hooks/pre-applypatch.sample
flyrank-ml-internship/.git/hooks/applypatch-msg.sample
flyrank-ml-internship/.git/hooks/fsmonitor-watchman.sample
flyrank-ml-internship/.git/hooks/pre-receive.sample
flyrank-ml-internship/.git/hooks/pre-rebase.sample
flyrank-ml-internship/.git/hooks/update.sample
flyrank-ml-internshi

In [96]:
import os

for root, dirs, files in os.walk("flyrank-ml-internship"):
    for file in files:
        print(os.path.join(root, file))

flyrank-ml-internship/CLAUDE.md
flyrank-ml-internship/LICENSE
flyrank-ml-internship/README.md
flyrank-ml-internship/index.md
flyrank-ml-internship/SETUP.md
flyrank-ml-internship/.gitignore
flyrank-ml-internship/DATA_USE.md
flyrank-ml-internship/requirements.txt
flyrank-ml-internship/GUIDE.md
flyrank-ml-internship/AGENTS.md
flyrank-ml-internship/.git/index
flyrank-ml-internship/.git/config
flyrank-ml-internship/.git/HEAD
flyrank-ml-internship/.git/packed-refs
flyrank-ml-internship/.git/description
flyrank-ml-internship/.git/hooks/pre-commit.sample
flyrank-ml-internship/.git/hooks/push-to-checkout.sample
flyrank-ml-internship/.git/hooks/commit-msg.sample
flyrank-ml-internship/.git/hooks/pre-applypatch.sample
flyrank-ml-internship/.git/hooks/applypatch-msg.sample
flyrank-ml-internship/.git/hooks/fsmonitor-watchman.sample
flyrank-ml-internship/.git/hooks/pre-receive.sample
flyrank-ml-internship/.git/hooks/pre-rebase.sample
flyrank-ml-internship/.git/hooks/update.sample
flyrank-ml-internshi

In [97]:
import os

for root, dirs, files in os.walk("flyrank-ml-internship"):
    for file in files:
        print(os.path.join(root, file))

flyrank-ml-internship/CLAUDE.md
flyrank-ml-internship/LICENSE
flyrank-ml-internship/README.md
flyrank-ml-internship/index.md
flyrank-ml-internship/SETUP.md
flyrank-ml-internship/.gitignore
flyrank-ml-internship/DATA_USE.md
flyrank-ml-internship/requirements.txt
flyrank-ml-internship/GUIDE.md
flyrank-ml-internship/AGENTS.md
flyrank-ml-internship/.git/index
flyrank-ml-internship/.git/config
flyrank-ml-internship/.git/HEAD
flyrank-ml-internship/.git/packed-refs
flyrank-ml-internship/.git/description
flyrank-ml-internship/.git/hooks/pre-commit.sample
flyrank-ml-internship/.git/hooks/push-to-checkout.sample
flyrank-ml-internship/.git/hooks/commit-msg.sample
flyrank-ml-internship/.git/hooks/pre-applypatch.sample
flyrank-ml-internship/.git/hooks/applypatch-msg.sample
flyrank-ml-internship/.git/hooks/fsmonitor-watchman.sample
flyrank-ml-internship/.git/hooks/pre-receive.sample
flyrank-ml-internship/.git/hooks/pre-rebase.sample
flyrank-ml-internship/.git/hooks/update.sample
flyrank-ml-internshi

In [98]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Dataset shape:", df.shape)
print(df["is_declining_label"].value_counts())

Dataset shape: (30000, 45)
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [99]:
import pandas as pd
import os

file_path = "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(file_path))

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

File exists: True
Dataset shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [100]:
import numpy as np

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [101]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))

print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]) & set(test["client_id"])

print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


In [102]:
!git clone https://github.com/Duchalsoham12/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


I chose Random Forest because the target is binary and the data contains both numeric and categorical fields. It can capture non-linear relationships and interactions without requiring a simple linear relationship. I will compare it with the existing hand-written baseline using the same metric and split.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [103]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))

print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]) & set(test["client_id"])

print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


I use a grouped-by-client 80/20 split. Content from the same client stays in only one split, so the model is tested on clients it did not train on. This is an honest split for measuring whether the model can generalize beyond the clients used for training.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [104]:
# Prepare features

excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

feature_cols = [
    c for c in df.columns
    if c not in excluded
]

X_train = train[feature_cols]
y_train = train["is_declining_label"]

X_test = test[feature_cols]
y_test = test["is_declining_label"]

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Total features:", len(feature_cols))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Total features: 40
Numeric features: 29
Categorical features: 11


In [105]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [106]:
pred_prob = model.predict_proba(X_test)[:, 1]

def precision_at_50(y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_n = min(50, len(y_true))
    top_indices = np.argsort(scores)[::-1][:top_n]

    return y_true[top_indices].mean()

model_p50 = precision_at_50(y_test, pred_prob)

print("Random Forest Precision@50:",
      round(model_p50, 3))

Random Forest Precision@50: 1.0


In [107]:
baseline_p50 = 0.700

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ]
})

display(comparison)

,method,precision_at_50
0,Week-4 baseline,0.7
1,Random Forest,1.0


I trained a Random Forest using the client-grouped split and evaluated it on the held-out clients. I use Precision@50 to compare the model with the Week-4 baseline using the same evaluation approach

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [108]:
# Error analysis

error_analysis = test[
    [
        "content_id",
        "trend_direction",
        "trend_pct",
        "content_age_days",
        "days_since_last_update",
        "engagement_rate",
        "ai_traffic_pct"
    ]
].copy()

error_analysis["predicted_probability"] = pred_prob

error_analysis["predicted_class"] = (
    error_analysis["predicted_probability"] >= 0.5
).astype(int)

error_analysis["actual_class"] = (
    test["is_declining_label"].values
)

error_analysis["correct"] = (
    error_analysis["predicted_class"]
    == error_analysis["actual_class"]
)

print("Test rows:", len(error_analysis))
print("Incorrect predictions:",
      int((~error_analysis["correct"]).sum()))

print("\nHighest-scored incorrect predictions:")

display(
    error_analysis[
        ~error_analysis["correct"]
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
    .head(10)
)

Test rows: 6163
Incorrect predictions: 1228

Highest-scored incorrect predictions:


,content_id,trend_direction,trend_pct,content_age_days,days_since_last_update,engagement_rate,ai_traffic_pct,predicted_probability,predicted_class,actual_class,correct
12332,content_4d9f36001f06,stable,-17.0,275,104,33.33,0.00,0.896667,1,0,False
2357,content_8f1409b2674e,stable,-17.9,271,104,0.00,0.00,0.890000,1,0,False
20736,content_41baf0722ad9,stable,-14.3,275,104,0.00,0.00,0.883333,1,0,False
10080,content_35d63627bf3e,stable,8.7,238,103,0.00,0.00,0.846667,1,0,False
5399,content_6677fd6c4ea5,stable,-13.7,280,104,0.00,0.00,0.843333,1,0,False
11061,content_0b47dae0c7f9,stable,-13.3,238,103,0.00,0.00,0.826667,1,0,False
1517,content_816d77e36e14,stable,-19.1,275,104,0.00,0.00,0.813333,1,0,False
1439,content_5585a0e7089c,stable,-8.7,223,102,0.00,0.00,0.810000,1,0,False
28582,content_f49660e074e9,stable,3.1,223,102,6.25,6.25,0.806667,1,0,False
22526,content_1d0963b56227,up,43.6,280,104,20.00,0.00,0.806667,1,0,False


In [109]:
print("ML-08 completed successfully.")
print("Train rows:", len(train))
print("Test rows:", len(test))
print("Client overlap:",
      len(set(train["client_id"]) & set(test["client_id"])))
print("Random Forest Precision@50:",
      round(model_p50, 3))

ML-08 completed successfully.
Train rows: 23837
Test rows: 6163
Client overlap: 0
Random Forest Precision@50: 1.0


In [110]:
print("Top 50 predicted probabilities:")
print(np.sort(pred_prob)[-50:][::-1])

print("\nActual positives in top 50:")
top_50_idx = np.argsort(pred_prob)[::-1][:50]
print(y_test.iloc[top_50_idx].sum())

print("\nTest positive rate:")
print(round(y_test.mean(), 3))

print("\nUnique prediction probabilities:")
print(len(np.unique(pred_prob)))

Top 50 predicted probabilities:
[0.97666667 0.97       0.96666667 0.96333333 0.96333333 0.96333333
 0.95666667 0.95666667 0.95333333 0.95       0.95       0.95
 0.95       0.94666667 0.94666667 0.94666667 0.94666667 0.94333333
 0.94333333 0.94333333 0.94333333 0.94       0.94       0.94
 0.93666667 0.93666667 0.93666667 0.93666667 0.93666667 0.93666667
 0.93666667 0.93333333 0.93333333 0.93333333 0.93       0.93
 0.92666667 0.92666667 0.92666667 0.92666667 0.92333333 0.92333333
 0.92333333 0.92       0.92       0.92       0.92       0.92
 0.91666667 0.91666667]

Actual positives in top 50:
50

Test positive rate:
0.511

Unique prediction probabilities:
291


In [111]:
import json

notebook_path = "flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        code = "".join(cell["source"])
        print(f"\n{'='*60}")
        print(f"CELL {i}")
        print(f"{'='*60}")
        print(code)


CELL 3
import pandas as pd
import numpy as np

file_path = "/content/flyrank-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

print("Data loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

CELL 4
# ML-07 Step 1: Signal checks

print("SIGNAL 1: trend_pct")
print("=" * 40)

trend_buckets = pd.cut(
    df["trend_pct"],
    bins=[-float("inf"), -50, -20, 0, float("inf")],
    labels=["Strong decline", "Moderate decline", "Flat", "Growth"]
)

trend_table = (
    df.assign(trend_bucket=trend_buckets)
      .groupby("trend_bucket", observed=False)
      .size()
      .reset_index(name="n")
)

display(trend_table)

print("\nSIGNAL 2: CTR")
print("=" * 40)

ctr_buckets = pd.cut(
    df["ctr"],
    bins=[-float("inf"), 0.25, 0.50, 1.00, float("inf")],
    labels=["Low", "Medium", "High", "Very high"]
)

ctr_table = (
    df.assign(ctr_bucket=ctr_buckets)
      .groupby("ctr_bucket", observed=False)
      .size()
      .reset_index(nam



I checked the model's highest-scored predictions and compared them with the observed labels. The errors show where the model can rank content incorrectly, so the model should be used for prioritization and decision-support rather than as a guaranteed prediction.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.